# 06 — Clustering

Side-by-side comparison of sklearn vs rustml KMeans and DBSCAN.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs, make_moons
import matplotlib.pyplot as plt

## KMeans on Gaussian Blobs

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

km = KMeans(n_clusters=3, random_state=42, n_init=10)
km.fit(X_blobs)

print(f'sklearn KMeans:')
print(f'  inertia: {km.inertia_:.2f}')
print(f'  n_iter:  {km.n_iter_}')
print(f'  centers: {km.cluster_centers_}')

plt.figure(figsize=(8, 6))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=km.labels_, cmap='viridis', s=30, alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], c='red', marker='X', s=200)
plt.title('sklearn KMeans (k=3)')
plt.show()

## DBSCAN on Moon-shaped Data

In [ ]:
X_moons, _ = make_moons(n_samples=300, noise=0.05, random_state=42)

db = DBSCAN(eps=0.2, min_samples=5)
db.fit(X_moons)

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise = list(db.labels_).count(-1)
print(f'sklearn DBSCAN:')
print(f'  clusters: {n_clusters}')
print(f'  noise points: {n_noise}')
print(f'  core samples: {len(db.core_sample_indices_)}')

plt.figure(figsize=(8, 6))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=db.labels_, cmap='viridis', s=30)
plt.title(f'sklearn DBSCAN (eps=0.2, min_samples=5) — {n_clusters} clusters')
plt.show()

## DBSCAN Epsilon Sensitivity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, eps in zip(axes, [0.1, 0.2, 0.5]):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_moons)
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='viridis', s=20)
    ax.set_title(f'eps={eps}, clusters={n_c}')
plt.suptitle('DBSCAN Epsilon Sensitivity')
plt.tight_layout()
plt.show()

## KMeans vs DBSCAN Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = km_moons.fit_predict(X_moons)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_labels, cmap='viridis', s=20)
axes[0].set_title('KMeans (k=2) — fails on non-convex shapes')

db_moons = DBSCAN(eps=0.2, min_samples=5)
db_labels = db_moons.fit_predict(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db_labels, cmap='viridis', s=20)
axes[1].set_title('DBSCAN (eps=0.2) — handles non-convex shapes')

plt.tight_layout()
plt.show()

## RustML Comparison (via PyO3 bindings)

Build the wheel first: `cd crates/rustml-python && maturin develop --release`

In [ ]:
try:
    import rustml_python as rml

    km_rml = rml.KMeans(n_clusters=3, max_iter=300, seed=42)
    km_rml.fit(X_blobs)
    print(f'rustml KMeans: inertia={km_rml.inertia:.2f}')

    db_rml = rml.Dbscan(eps=0.2, min_samples=5)
    db_rml.fit(X_moons)
    print(f'rustml DBSCAN: clusters={db_rml.n_clusters}')
except ImportError:
    print('rustml_python not installed — run: cd crates/rustml-python && maturin develop --release')